# Flipkart Reviews — Preprocessing NLP
## Auteure : Ihssane Moutchou | PFA 4ème Année
## Objectif : Nettoyer le texte des avis et préparer
## les features TF-IDF pour l'entraînement des modèles.
## Entrée  : data/flipkart_data_with_sentiment.csv (produit par Aymen)
## Sortie  : data/flipkart_data_preprocessed.csv

In [ ]:
!pip install nltk -q
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import joblib, warnings
warnings.filterwarnings('ignore')
print("Imports OK")

In [ ]:
# On charge le CSV que Aymen a produit dans sa Phase 1
# ATTENTION : les sentiments sont 'Positif'/'Négatif'/'Neutre'
# avec majuscule et accents — on va les normaliser
df = pd.read_csv('/content/drive/MyDrive/flipkart_sentiments_pfa/data/flipkart_data_with_sentiment.csv')

print(f"Dataset chargé : {df.shape[0]:,} avis")
print("Colonnes :", df.columns.tolist())
print("Sentiments :", df['sentiment'].unique())
print("Valeurs manquantes :\n", df.isnull().sum())
df.head(3)

In [ ]:
# Normaliser : 'Positif' → 'POSITIF' etc. pour cohérence
mapping = {
    'Positif': 'POSITIF',
    'Négatif': 'NEGATIF',
    'Neutre':  'NEUTRE'
}
df['sentiment'] = df['sentiment'].map(mapping)

# Supprimer les lignes avec review vide
df = df.dropna(subset=['review'])
df['review'] = df['review'].astype(str)

print("Après normalisation :")
print(df['sentiment'].value_counts())

In [ ]:
# Pipeline NLP : lowercase → regex → tokenize → stopwords → lemmatize
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def clean_text(text):
    # 1. Minuscules
    text = text.lower()
    # 2. Supprimer ponctuation et chiffres
    text = re.sub(r'[^a-z\s]', '', text)
    # 3. Tokeniser
    tokens = word_tokenize(text)
    # 4. Supprimer stopwords + mots courts
    tokens = [t for t in tokens
              if t not in stop_words and len(t) > 2]
    # 5. Lemmatiser
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Test rapide
exemple = "The product is really good! I loved it and bought 2 pieces."
print("Avant :", exemple)
print("Après :", clean_text(exemple))

In [ ]:
# Appliquer sur tout le dataset (peut prendre 1–2 min)
print("Nettoyage en cours...")
df['review_clean'] = df['review'].apply(clean_text)
print(f"Terminé — {len(df):,} avis nettoyés")

# Vérification : comparer avant/après
for _, row in df.sample(3, random_state=42).iterrows():
    print(f"\n[{row['sentiment']}]")
    print("Avant :", row['review'][:80])
    print("Après :", row['review_clean'][:80])

In [ ]:
# TF-IDF : texte → vecteurs numériques
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X = tfidf.fit_transform(df['review_clean'])
y = df['sentiment']

print(f"Matrice TF-IDF : {X.shape[0]:,} avis × {X.shape[1]:,} features")

# Split 80% train / 20% test — graine fixe pour reproductibilité
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train : {X_train.shape[0]:,} avis")
print(f"Test  : {X_test.shape[0]:,} avis")

# Distribution des classes dans train
print("\nDistribution train :")
print(pd.Series(y_train).value_counts())

In [ ]:
# Sauvegarder le vectorizer TF-IDF (sera chargé dans les modèles)
joblib.dump(tfidf, '/content/drive/MyDrive/flipkart_sentiments_pfa/models/tfidf_vectorizer.pkl')
print("tfidf_vectorizer.pkl sauvegardé dans models/")

# Sauvegarder le CSV nettoyé pour Aymen (Phase 3)
df[['review','review_clean','sentiment']].to_csv(
    '/content/drive/MyDrive/flipkart_sentiments_pfa/data/flipkart_data_preprocessed.csv', index=False
)
print("flipkart_data_preprocessed.csv sauvegardé")

# Résumé final
print("\n"+"="*50)
print("SYNTHÈSE PREPROCESSING")
print("="*50)
print(f"Avis traités     : {len(df):,}")
print(f"Features TF-IDF  : 5 000")
print(f"Train / Test     : {X_train.shape[0]:,} / {X_test.shape[0]:,}")
print("Preprocessing terminé — prêt pour la Phase 3 !")

In [ ]:
import joblib
import pandas as pd

# Sauvegarder les données d'entraînement et de test
joblib.dump(X_train, '/content/drive/MyDrive/flipkart_sentiments_pfa/data/X_train.pkl')
joblib.dump(X_test, '/content/drive/MyDrive/flipkart_sentiments_pfa/data/X_test.pkl')
y_train.to_csv('/content/drive/MyDrive/flipkart_sentiments_pfa/data/y_train.csv', index=False)
y_test.to_csv('/content/drive/MyDrive/flipkart_sentiments_pfa/data/y_test.csv', index=False)

print("X_train.pkl, X_test.pkl, y_train.csv, y_test.csv sauvegardés dans data/")